# DeepSentinel — Training on Kaggle


In [ ]:
# Step 1 — Install extra dependencies (torch/torchvision already on Kaggle)
!pip install -q timm albumentations scikit-learn

In [ ]:
# Confirm GPU and locate dataset
import torch
from pathlib import Path

assert torch.cuda.is_available(), "GPU not enabled! Right panel → Accelerator → GPU T4 x2"
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

SEARCH_ROOTS = [
    Path('/kaggle/input/datasets/pranabr0y/celebdf-v2image-dataset/Celeb_V2'),
    Path('/kaggle/input/celebdf-v2image-dataset/Celeb_V2'),
    Path('/kaggle/input/celebdf-v2image-dataset'),
    Path('/kaggle/input/celeb-df-v2/Celeb_V2'),
    Path('/kaggle/input/celeb-df-v2'),
]
DATA_ROOT = None
for p in SEARCH_ROOTS:
    if (p / 'Train').exists():
        DATA_ROOT = p
        break

assert DATA_ROOT is not None, (
    "Dataset not found! Right panel → + Add Data → search celebdf-v2image-dataset → Add.\n"
    f"Searched: {[str(p) for p in SEARCH_ROOTS]}"
)
print(f"Dataset root: {DATA_ROOT}")

for split in ['Train', 'Val', 'Test']:
    for label in ['real', 'fake']:
        p = DATA_ROOT / split / label
        n = len(list(p.glob('*'))) if p.exists() else 0
        print(f"  {split}/{label}: {n:,} images")

In [ ]:
# Model definition
import torch.nn as nn
import timm

class DeepfakeDetector(nn.Module):
    def __init__(self, model_name='efficientnet_b4', pretrained=True):
        super().__init__()
        self.backbone = timm.create_model(model_name, pretrained=pretrained, num_classes=0)
        self.classifier = nn.Sequential(
            nn.Dropout(p=0.3),
            nn.Linear(self.backbone.num_features, 1),
        )

    def forward(self, x):
        return self.classifier(self.backbone(x))

print('Model defined OK')

In [ ]:
# Data loaders
import torchvision.transforms as T
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader

IMAGE_SIZE = 224
BATCH_SIZE = 64

train_transforms = T.Compose([
    T.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    T.RandomHorizontalFlip(),
    T.ColorJitter(brightness=0.2, contrast=0.2),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

infer_transforms = T.Compose([
    T.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

train_ds = ImageFolder(str(DATA_ROOT / 'Train'), transform=train_transforms)
val_ds   = ImageFolder(str(DATA_ROOT / 'Val'),   transform=infer_transforms)

print('Classes:', train_ds.classes)
print(f'Train: {len(train_ds):,}  |  Val: {len(val_ds):,}')

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=4, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)
print('DataLoaders ready')

In [ ]:
# Train  (auto-uploads to HuggingFace after each best epoch)
# Setting HuggingFace token and username before running
HF_TOKEN    = ""   # get from huggingface.co → Settings → Access Tokens (Write)
HF_USERNAME = ""   # your HuggingFace username

import logging, os, threading
import torch.nn as nn

logging.basicConfig(format='%(asctime)s | %(message)s', level=logging.INFO)
logger = logging.getLogger()

DEVICE      = 'cuda'
EPOCHS      = 10
LR          = 1e-4
WEIGHTS_OUT = '/kaggle/working/efficientnet_b4.pth'

os.makedirs('/kaggle/working', exist_ok=True)

model     = DeepfakeDetector(pretrained=True).to(DEVICE)
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=LR)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

def upload_to_hf(path):
    try:
        from huggingface_hub import HfApi
        api = HfApi()
        repo_id = f"{HF_USERNAME}/deepsentinel-weights"
        api.create_repo(repo_id, repo_type="model", exist_ok=True, token=HF_TOKEN)
        api.upload_file(
            path_or_fileobj=path,
            path_in_repo="efficientnet_b4.pth",
            repo_id=repo_id,
            repo_type="model",
            token=HF_TOKEN,
        )
        print(f"\n*** UPLOADED! Download from: https://huggingface.co/{repo_id} ***\n")
    except Exception as e:
        print(f"Upload failed: {e} — file still at {path}")

def run_val(model, loader):
    model.eval()
    total = 0.0
    with torch.no_grad():
        for imgs, labels in loader:
            imgs   = imgs.to(DEVICE)
            labels = labels.float().unsqueeze(1).to(DEVICE)
            total += criterion(model(imgs), labels).item()
    return total / len(loader)

best_val_loss = float('inf')

for epoch in range(1, EPOCHS + 1):
    model.train()
    train_loss = 0.0
    for imgs, labels in train_loader:
        imgs   = imgs.to(DEVICE)
        labels = labels.float().unsqueeze(1).to(DEVICE)
        optimizer.zero_grad()
        loss = criterion(model(imgs), labels)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()

    val_loss = run_val(model, val_loader)
    scheduler.step()
    logger.info(f'Epoch {epoch}/{EPOCHS}  train={train_loss/len(train_loader):.4f}  val={val_loss:.4f}')

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), WEIGHTS_OUT)
        logger.info(f'  -> saved best (val={val_loss:.4f}) — uploading to HuggingFace...')
        if HF_TOKEN and HF_USERNAME:
            threading.Thread(target=upload_to_hf, args=(WEIGHTS_OUT,), daemon=True).start()
        else:
            print("HF_TOKEN or HF_USERNAME not set — skipping auto-upload")

print(f'\nDone! Best val_loss: {best_val_loss:.4f}')
print(f'Weights at: {WEIGHTS_OUT}')

In [ ]:
# Evaluate on test set
import numpy as np
from sklearn.metrics import classification_report, roc_auc_score

test_ds     = ImageFolder(str(DATA_ROOT / 'Test'), transform=infer_transforms)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=4)

model.load_state_dict(torch.load(WEIGHTS_OUT, map_location=DEVICE))
model.eval()

all_probs, all_labels = [], []
with torch.no_grad():
    for imgs, labels in test_loader:
        probs = torch.sigmoid(model(imgs.to(DEVICE))).squeeze(1).cpu().numpy()
        all_probs.extend(probs)
        all_labels.extend(labels.numpy())

preds = (np.array(all_probs) >= 0.5).astype(int)
auc   = roc_auc_score(all_labels, all_probs)

print(f'Test AUC: {auc:.4f}')
print(classification_report(all_labels, preds, target_names=test_ds.classes))

In [ ]:
import os
size_mb = os.path.getsize(WEIGHTS_OUT) / 1e6
print(f'Weights file: {WEIGHTS_OUT}  ({size_mb:.1f} MB)')
print()
print('Next steps:')
print('  1. Right panel → Output tab → download efficientnet_b4.pth')
print('  2. Place it at: C:\\Users\\Hp\\DeepSentinel\\models\\weights\\efficientnet_b4.pth')
print('  3. Run: python app/gradio_app.py')